In [3]:
!pip install torch torchvision torchaudio
!pip install torch-geometric scikit-learn matplotlib seaborn pandas umap-learn

   ---------------------------------------- 0.0/241.3 MB ? eta -:--:--
   ---------------------------------------- 0.1/241.3 MB 3.6 MB/s eta 0:01:07
   ---------------------------------------- 0.2/241.3 MB 2.9 MB/s eta 0:01:24
   ---------------------------------------- 0.4/241.3 MB 3.0 MB/s eta 0:01:20
   ---------------------------------------- 0.5/241.3 MB 2.9 MB/s eta 0:01:25
   ---------------------------------------- 0.6/241.3 MB 2.7 MB/s eta 0:01:31
   ---------------------------------------- 0.7/241.3 MB 2.8 MB/s eta 0:01:26
   ---------------------------------------- 0.8/241.3 MB 2.8 MB/s eta 0:01:28
   ---------------------------------------- 0.9/241.3 MB 2.6 MB/s eta 0:01:31
   ---------------------------------------- 1.0/241.3 MB 2.4 MB/s eta 0:01:39
   ---------------------------------------- 1.0/241.3 MB 2.4 MB/s eta 0:01:43
   ---------------------------------------- 1.1/241.3 MB 2.3 MB/s eta 0:01:45
   ---------------------------------------- 1.2/241.3 MB 2.2 MB/s eta 0

In [11]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GATConv, SAGEConv
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import umap
import pandas as pd
import os

In [13]:
# ------------------------------
# 1. Define Models
# ------------------------------
class GCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GCN, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, heads=8):
        super(GAT, self).__init__()
        self.conv1 = GATConv(in_channels, hidden_channels, heads=heads, dropout=0.6)
        self.conv2 = GATConv(hidden_channels * heads, out_channels, heads=1, concat=False, dropout=0.6)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)


class GraphSAGE(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels):
        super(GraphSAGE, self).__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, out_channels)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index).relu()
        x = F.dropout(x, p=0.5, training=self.training)
        return self.conv2(x, edge_index)

In [15]:
# ------------------------------
# 2. Training and Evaluation Function
# ------------------------------
def train_and_evaluate(dataset_name, ModelClass):
    dataset = Planetoid(root=f"data/{dataset_name}", name=dataset_name)
    data = dataset[0]
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model = ModelClass(dataset.num_node_features, 16, dataset.num_classes).to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)

    # Train
    model.train()
    for epoch in range(200):
        optimizer.zero_grad()
        out = model(data.x, data.edge_index)
        loss = F.cross_entropy(out[data.train_mask], data.y[data.train_mask])
        loss.backward()
        optimizer.step()

    # Evaluate
    model.eval()
    out = model(data.x, data.edge_index)
    preds = out.argmax(dim=1).cpu()
    y_true = data.y.cpu()

    acc = accuracy_score(y_true[data.test_mask], preds[data.test_mask])
    f1_macro = f1_score(y_true[data.test_mask], preds[data.test_mask], average="macro")
    f1_weighted = f1_score(y_true[data.test_mask], preds[data.test_mask], average="weighted")

    # Confusion Matrix
    cm = confusion_matrix(y_true[data.test_mask], preds[data.test_mask])
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.title(f"{ModelClass.__name__} on {dataset_name} - Confusion Matrix")
    plt.savefig(f"results/{dataset_name}_{ModelClass.__name__}_cm.png", dpi=300)
    plt.close()

    # Embedding Visualization with UMAP
    reducer = umap.UMAP()
    embedding = reducer.fit_transform(out.detach().cpu().numpy())
    plt.figure(figsize=(6, 5))
    plt.scatter(embedding[:, 0], embedding[:, 1], c=y_true, cmap="tab10", s=10)
    plt.title(f"{ModelClass.__name__} on {dataset_name} - Embedding Space")
    plt.savefig(f"results/{dataset_name}_{ModelClass.__name__}_embedding.png", dpi=300)
    plt.close()

    return acc, f1_macro, f1_weighted


In [17]:
# ------------------------------
# 3. Run Experiments
# ------------------------------
if __name__ == "__main__":
    import os
    os.makedirs("results", exist_ok=True)

    datasets = ["Cora", "CiteSeer"]
    models = [GCN, GAT, GraphSAGE]

    print("=== Graph Neural Networks Comparative Study ===\n")
    results = []

    for dataset in datasets:
        for model in models:
            acc, f1_macro, f1_weighted = train_and_evaluate(dataset, model)
            results.append([dataset, model.__name__, acc, f1_macro, f1_weighted])
            print(f"{dataset} - {model.__name__} | Acc: {acc:.4f}, F1-macro: {f1_macro:.4f}, F1-weighted: {f1_weighted:.4f}")

    # Save final results table
    import pandas as pd
    df = pd.DataFrame(results, columns=["Dataset", "Model", "Accuracy", "F1-macro", "F1-weighted"])
    df.to_csv("results/summary.csv", index=False)
    print("\nResults saved in results/summary.csv and plots in /results/")

=== Graph Neural Networks Comparative Study ===

Cora - GCN | Acc: 0.8080, F1-macro: 0.8027, F1-weighted: 0.8089
Cora - GAT | Acc: 0.7930, F1-macro: 0.7892, F1-weighted: 0.7954
Cora - GraphSAGE | Acc: 0.8010, F1-macro: 0.7912, F1-weighted: 0.8017


Processing...
Done!


CiteSeer - GCN | Acc: 0.6870, F1-macro: 0.6547, F1-weighted: 0.6887
CiteSeer - GAT | Acc: 0.6900, F1-macro: 0.6606, F1-weighted: 0.6987
CiteSeer - GraphSAGE | Acc: 0.6830, F1-macro: 0.6571, F1-weighted: 0.6921

Results saved in results/summary.csv and plots in /results/


In [19]:
# ------------------------------
# 4. Results Table
# ------------------------------
df = pd.DataFrame(results, columns=["Dataset", "Model", "Accuracy", "F1-macro", "F1-weighted"])
df


,Dataset,Model,Accuracy,F1-macro,F1-weighted
0,Cora,GCN,0.808,0.802698,0.808869
1,Cora,GAT,0.793,0.789161,0.795424
2,Cora,GraphSAGE,0.801,0.791158,0.801679
3,CiteSeer,GCN,0.687,0.654745,0.688686
4,CiteSeer,GAT,0.690,0.660551,0.698728
5,CiteSeer,GraphSAGE,0.683,0.657122,0.692107
